# Barcelona point cloud → individual tree wraps

Pipeline:
1. **Inspect** — profile all classification codes across tiles to confirm class 5 is present
2. **Per-tile loop** — for each tile: filter class 5 → SOR → voxel downsample → Li 2012 segmentation → alpha-wrap each tree → save one combined `.obj` per tile

Only one tile is in memory at a time. Tree segmentation uses Li et al. (2012) — top-down, point-based, no CHM rasterisation.

In [1]:
import math
import re
import time
from pathlib import Path

import laspy
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import KDTree
from scipy.ndimage import gaussian_filter, distance_transform_edt

from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.measure import marching_cubes

## Configuration
Edit these before running.

In [2]:
TILES_DIR  = Path("bcn_new_pointcloud")
OUTPUT_DIR = Path("bcn_canopy_objs_Li")
VIZ_DIR    = OUTPUT_DIR / "segmentation_viz"

# Per-tree meshing parameters (Marching Cubes)
MC_VOXEL_SIZE = 0.75
MC_SIGMA      = 1.5
MC_ISO_LEVEL  = 0.15

# Class codes to treat as high vegetation
HIGH_VEG_CLASSES = [5]
NDVI_THRESHOLD   = 0.2

# Translation subtracted from all output coordinates
TRANSLATION = (429692.0, 4582890.0, 0.0)

# --- Point cloud cleaning (vegetation only) ---
MULTI_RETURN_ONLY = True
SOR_NEIGHBORS     = 20
SOR_STD_RATIO     = 1.5
VOXEL_SIZE        = 1.0

# --- CHM-based tree segmentation ---
CHM_CELL_SIZE      = 0.5   # m — grid resolution for DTM/DSM/CHM
CHM_SMOOTH_SIGMA   = 2.0   # grid cells — Gaussian smooth before peak detection
CHM_MIN_HEIGHT     = 3.0   # m — excludes rooftop shrubs and low vegetation
CHM_PEAK_MIN_DIST  = 5.0   # m — minimum distance between tree tops
CHM_MIN_TREE_PTS   = 150   # discard segments with fewer vegetation points

# --- Trunk modeling ---
# DBH = TRUNK_ALLOM_A * H ^ TRUNK_ALLOM_B   (H and DBH in metres)
# Default coefficients are a generic mixed-species fit; replace with
# species-specific values if the Barcelona street-tree inventory is available.
TRUNK_Q_BASE   = 0.20   # quantile of crown z used for trunk-base centroid
TRUNK_Q_CROWN  = 0.10   # quantile of crown z used as crown-base elevation
TRUNK_ALLOM_A  = 0.03  # allometric coefficient a
TRUNK_ALLOM_B  = 1.2    # allometric exponent b
TRUNK_TAPER    = 0.4    # r_top / r_base
TRUNK_N_SIDES  = 12     # polygon sides per trunk cone

OUTPUT_DIR.mkdir(exist_ok=True)
VIZ_DIR.mkdir(exist_ok=True)

---
## Step 1 — Inspect: what classes are in the tiles?

In [3]:
# Catalan LiDAR classification scheme (ICGC)
ASPRS = {
    1:  "Default (no classificats)",
    2:  "Ground (terreny)",
    3:  "Low vegetation (vegetació baixa)",
    4:  "Medium vegetation (vegetació mitjana)",
    5:  "High vegetation (vegetació alta)",
    6:  "Building (edificis)",
    7:  "Low points (punts baixos)",
    8:  "Model key points (punts clau)",
    9:  "Water (aigua)",
    12: "Overlapping (solapament)",
    17: "Bridge (ponts)",
    18: "Air points (punts aeris)",
    75: "Other ground (possible terreny)",
    76: "Under building (sota edificis)",
    77: "Other building (damunt edificis)",
}

tiles = sorted(TILES_DIR.glob("*.laz")) + sorted(TILES_DIR.glob("*.las"))
print(f"Found {len(tiles)} tiles")

# Aggregate per-class statistics across all tiles
class_stats = {}   # class_code → {count, z_min, z_max, ndvi_sum, multi_return_count}

for tile in tiles:
    with laspy.open(tile) as f:
        las = f.read()

    has_nir = hasattr(las, "nir")
    z    = np.asarray(las.z,                 dtype=np.float32)
    ret  = np.asarray(las.number_of_returns, dtype=np.uint8)
    cls  = np.asarray(las.classification,    dtype=np.uint8)

    if has_nir:
        nir  = np.asarray(las.nir, dtype=np.float64)
        red  = np.asarray(las.red, dtype=np.float64)
        ndvi = (nir - red) / (nir + red + 1e-8)

    for code_ in np.unique(cls):
        mask = cls == code_
        s = class_stats.setdefault(int(code_), {
            "count": 0, "z_min": np.inf, "z_max": -np.inf,
            "ndvi_sum": 0.0, "multi_return": 0
        })
        s["count"]        += int(mask.sum())
        s["z_min"]         = min(s["z_min"], float(z[mask].min()))
        s["z_max"]         = max(s["z_max"], float(z[mask].max()))
        s["multi_return"] += int((ret[mask] > 1).sum())
        if has_nir:
            s["ndvi_sum"]  += float(ndvi[mask].sum())

print(f"\n{'Code':>6}  {'Name':<38}  {'Points':>12}  {'Z range':>16}  {'NDVI mean':>10}  {'Multi-return':>14}")
print("-" * 110)
for code_, s in sorted(class_stats.items()):
    n         = s["count"]
    ndvi_mean = s["ndvi_sum"] / n if n else 0
    multi_pct = s["multi_return"] / n * 100 if n else 0
    name      = ASPRS.get(code_, "(unknown — not in Catalan scheme)")
    print(f"{code_:>6}  {name:<38}  {n:>12,}  "
          f"[{s['z_min']:5.1f}, {s['z_max']:6.1f}]  "
          f"{ndvi_mean:>10.3f}  {multi_pct:>13.1f}%")

Found 25 tiles

  Code  Name                                          Points           Z range   NDVI mean    Multi-return
--------------------------------------------------------------------------------------------------------------
     1  Default (no classificats)                 14,214,171  [ -9.9,  195.4]      -0.009           40.7%
     2  Ground (terreny)                          36,173,162  [-10.1,  186.3]       0.097           21.0%
     3  Low vegetation (vegetació baixa)           4,714,886  [ -9.5,  187.4]       0.273           57.4%
     4  Medium vegetation (vegetació mitjana)      2,301,693  [  0.7,  188.3]       0.301           75.2%
     5  High vegetation (vegetació alta)          67,726,959  [ -1.4,  193.5]       0.355           72.8%
     6  Building (edificis)                       97,299,153  [ -8.0,  195.0]       0.041           16.2%
     7  Low points (punts baixos)                     74,755  [-12.8,  176.7]      -0.044           67.8%
     8  Model key points

**Read the table above**, then fill in `HIGH_VEG_CLASSES` in the configuration cell.

High-vegetation points typically have:
- multi-return rate > 80 % (pulses that penetrate the canopy)
- NDVI > 0.2
- z range spanning the full height (both canopy top and intermediate hits)

If your data uses standard ASPRS codes, class 5 = High Vegetation.  
If not, identify the code(s) that match the criteria above and set `HIGH_VEG_CLASSES = [...]`.

Leave `HIGH_VEG_CLASSES = []` to fall back to NDVI-based filtering.

---
## Step 2 — Helper functions: filter, clean, segment, wrap

In [4]:
def filter_tile(las):
    """Class 5 / NDVI filter + multi-return filter. Returns xyz array."""
    if HIGH_VEG_CLASSES:
        mask = np.isin(np.asarray(las.classification, dtype=np.uint8), HIGH_VEG_CLASSES)
    else:
        nir  = np.asarray(las.nir, dtype=np.float64)
        red  = np.asarray(las.red, dtype=np.float64)
        ndvi = (nir - red) / (nir + red + 1e-8)
        mask = ndvi > NDVI_THRESHOLD
    if MULTI_RETURN_ONLY:
        mask &= np.asarray(las.number_of_returns) > 1
    return np.column_stack((las.x[mask], las.y[mask], las.z[mask]))


def filter_tile_ground(las):
    """Return class 2 (ground) points as xyz array."""
    mask = np.asarray(las.classification, dtype=np.uint8) == 2
    return np.column_stack((las.x[mask], las.y[mask], las.z[mask]))


def remove_outliers_sor(pts):
    tree = KDTree(pts)
    dists, _ = tree.query(pts, k=SOR_NEIGHBORS + 1, workers=-1)
    mean_dists = dists[:, 1:].mean(axis=1)
    threshold  = mean_dists.mean() + SOR_STD_RATIO * mean_dists.std()
    return pts[mean_dists < threshold]


def voxel_downsample(pts, voxel_size):
    voxel_ids = np.floor(pts / voxel_size).astype(np.int32)
    _, unique_idx = np.unique(voxel_ids, axis=0, return_index=True)
    return pts[unique_idx]


# ---------------------------------------------------------------------------
# CHM segmentation
# ---------------------------------------------------------------------------

def segment_trees_chm(pts_veg, pts_ground, cell_size=0.5, smooth_sigma=2.0,
                      min_height=2.0, peak_min_dist_m=3.0, min_tree_points=80):
    """
    CHM-based individual tree segmentation.

    Returns
    -------
    labels      : (N,) int32 — 0 = unassigned; 1..K = tree IDs
    dtm         : (ny, nx) float64 — Digital Terrain Model raster
    (x_min, y_min, cell_size) : georeferencing of the raster
    """
    x_min = min(pts_veg[:, 0].min(), pts_ground[:, 0].min())
    y_min = min(pts_veg[:, 1].min(), pts_ground[:, 1].min())
    x_max = max(pts_veg[:, 0].max(), pts_ground[:, 0].max())
    y_max = max(pts_veg[:, 1].max(), pts_ground[:, 1].max())

    nx = int(np.ceil((x_max - x_min) / cell_size)) + 1
    ny = int(np.ceil((y_max - y_min) / cell_size)) + 1

    def to_rc(pts):
        c = np.clip(((pts[:, 0] - x_min) / cell_size).astype(int), 0, nx - 1)
        r = np.clip(((pts[:, 1] - y_min) / cell_size).astype(int), 0, ny - 1)
        return r, c

    # DSM: max vegetation Z per cell
    flat_dsm = np.full(ny * nx, -np.inf, dtype=np.float64)
    r_v, c_v = to_rc(pts_veg)
    np.maximum.at(flat_dsm, r_v * nx + c_v, pts_veg[:, 2])
    dsm = flat_dsm.reshape(ny, nx)
    dsm[dsm == -np.inf] = np.nan

    # DTM: mean ground Z per cell, gaps filled by nearest valid cell
    dtm_sum = np.zeros(ny * nx, dtype=np.float64)
    dtm_cnt = np.zeros(ny * nx, dtype=np.int32)
    r_g, c_g = to_rc(pts_ground)
    np.add.at(dtm_sum, r_g * nx + c_g, pts_ground[:, 2])
    np.add.at(dtm_cnt, r_g * nx + c_g, 1)
    dtm = np.where(dtm_cnt > 0,
                   dtm_sum / np.where(dtm_cnt > 0, dtm_cnt, 1),
                   np.nan).reshape(ny, nx)
    nan_mask = np.isnan(dtm)
    if nan_mask.any():
        fill_idx = distance_transform_edt(nan_mask, return_distances=False,
                                          return_indices=True)
        dtm = dtm[tuple(fill_idx)]

    # CHM
    chm = np.where(~np.isnan(dsm), dsm - dtm, 0.0)
    chm = np.maximum(chm, 0.0)
    chm_masked = np.where(chm >= min_height, chm, 0.0)
    chm_smooth = gaussian_filter(chm_masked, sigma=smooth_sigma)

    # Tree tops: local maxima in smoothed CHM
    min_dist_cells = max(1, int(peak_min_dist_m / cell_size))
    peaks = peak_local_max(chm_smooth, min_distance=min_dist_cells,
                           labels=(chm_smooth > 0))

    if len(peaks) == 0:
        print("    CHM: no peaks found — check CHM_MIN_HEIGHT or point density")
        return np.zeros(len(pts_veg), dtype=np.int32), dtm, (x_min, y_min, cell_size)

    markers = np.zeros((ny, nx), dtype=np.int32)
    markers[peaks[:, 0], peaks[:, 1]] = np.arange(1, len(peaks) + 1)
    labels_2d = watershed(-chm_smooth, markers, mask=(chm_smooth > 0))

    labels = labels_2d[r_v, c_v].astype(np.int32)

    # Drop tiny segments
    unique_lbls, counts = np.unique(labels, return_counts=True)
    for lbl, cnt in zip(unique_lbls, counts):
        if lbl > 0 and cnt < min_tree_points:
            labels[labels == lbl] = 0

    n_trees = int((np.unique(labels) > 0).sum())
    print(f"    CHM: {len(peaks)} peaks → {n_trees} trees (≥ {min_tree_points} pts)")
    return labels, dtm, (x_min, y_min, cell_size)


# ---------------------------------------------------------------------------
# Trunk geometry helpers
# ---------------------------------------------------------------------------

def sample_dtm(dtm, x_min, y_min, cell_size, x, y):
    """
    Bilinear sample of the DTM raster at world coordinates (x, y).
    x, y can be scalars or arrays. Returns z_ground. Clamps to raster bounds.
    """
    ny, nx = dtm.shape
    fx = (np.asarray(x) - x_min) / cell_size
    fy = (np.asarray(y) - y_min) / cell_size
    fx = np.clip(fx, 0, nx - 1.001)
    fy = np.clip(fy, 0, ny - 1.001)
    x0 = np.floor(fx).astype(int);  x1 = x0 + 1
    y0 = np.floor(fy).astype(int);  y1 = y0 + 1
    wx = fx - x0;  wy = fy - y0
    return ((1 - wx) * (1 - wy) * dtm[y0, x0]
          +      wx  * (1 - wy) * dtm[y0, x1]
          + (1 - wx) *      wy  * dtm[y1, x0]
          +      wx  *      wy  * dtm[y1, x1])


def estimate_trunk_params(tree_pts, dtm, x_min, y_min, cell_size,
                          q_base=0.20, q_cb=0.10,
                          allom_a=0.012, allom_b=1.2,
                          taper_ratio=0.4):
    """
    Estimate trunk geometry for one segmented crown.

    Uses the lowest q_base fraction of crown points to locate the trunk base
    in XY, looks up DTM for ground elevation, then fits a tapered cone from
    ground to crown-base using an allometric DBH model.

    Returns a dict with trunk geometry, or None if geometry is implausible.
    """
    if len(tree_pts) < 20:
        return None

    z = tree_pts[:, 2]
    z_thresh = np.quantile(z, q_base)
    low_mask  = z <= z_thresh
    if low_mask.sum() < 3:
        return None

    x_t = float(tree_pts[low_mask, 0].mean())
    y_t = float(tree_pts[low_mask, 1].mean())

    z_base       = float(sample_dtm(dtm, x_min, y_min, cell_size, x_t, y_t))
    z_crown_base = float(np.quantile(z, q_cb))
    # Minimum trunk height of 1.5 m above ground (handles tall narrow crowns
    # like cypresses where the 10th-percentile point is near ground level)
    z_crown_base = max(z_crown_base, z_base + 1.5)
    z_top        = float(z.max())

    h_trunk = z_crown_base - z_base
    h_total = z_top - z_base
    if h_trunk <= 0.3 or h_total <= 1.0:
        return None

    dbh    = allom_a * (h_total ** allom_b)
    r_base = dbh / 2.0
    r_top  = taper_ratio * r_base

    return {
        "x": x_t, "y": y_t,
        "z_base": z_base, "z_crown_base": z_crown_base, "z_top": z_top,
        "height_total": h_total, "height_trunk": h_trunk,
        "dbh": dbh, "r_base": r_base, "r_top": r_top,
    }


def make_trunk_mesh(x, y, z_base, z_top, r_base, r_top,
                    n_sides=12, close_caps=True):
    """
    Closed truncated cone aligned with +Z.

    Returns verts (V, 3) and faces (F, 3).
    Normals are outward-facing; mesh is watertight when close_caps=True.
    """
    theta = np.linspace(0, 2 * np.pi, n_sides, endpoint=False)
    cos_t = np.cos(theta)
    sin_t = np.sin(theta)

    ring_bot = np.column_stack([x + r_base * cos_t,
                                y + r_base * sin_t,
                                np.full(n_sides, z_base)])
    ring_top = np.column_stack([x + r_top * cos_t,
                                y + r_top * sin_t,
                                np.full(n_sides, z_top)])
    verts = [ring_bot, ring_top]

    faces = []
    for k in range(n_sides):
        j = (k + 1) % n_sides
        faces.append([k,            j,            j + n_sides])
        faces.append([k,            j + n_sides,  k + n_sides])

    if close_caps:
        c_bot = np.array([[x, y, z_base]])
        c_top = np.array([[x, y, z_top]])
        c_bot_idx = 2 * n_sides
        c_top_idx = 2 * n_sides + 1
        verts.extend([c_bot, c_top])
        for k in range(n_sides):
            j = (k + 1) % n_sides
            faces.append([c_bot_idx, j,            k           ])  # bottom cap (-Z normal)
            faces.append([c_top_idx, k + n_sides,  j + n_sides ])  # top    cap (+Z normal)

    return np.vstack(verts), np.array(faces, dtype=np.int32)


# ---------------------------------------------------------------------------
# Visualisation
# ---------------------------------------------------------------------------

def save_segmentation_image(pts_veg, labels, tile_name, viz_dir, cell_size=1.0):
    """Top-down 2D map: each tree a different colour. Saved as PNG."""
    x_min, y_min = pts_veg[:, :2].min(axis=0)
    x_max, y_max = pts_veg[:, :2].max(axis=0)
    nx = int(np.ceil((x_max - x_min) / cell_size)) + 1
    ny = int(np.ceil((y_max - y_min) / cell_size)) + 1

    c = np.clip(((pts_veg[:, 0] - x_min) / cell_size).astype(int), 0, nx - 1)
    r = np.clip(((pts_veg[:, 1] - y_min) / cell_size).astype(int), 0, ny - 1)
    label_raster = np.zeros((ny, nx), dtype=np.int32)
    label_raster[r, c] = labels

    n_lbls = int(label_raster.max()) + 1
    rng = np.random.default_rng(42)
    palette = rng.integers(60, 230, size=(n_lbls, 3), dtype=np.uint8)
    palette[0] = [25, 25, 25]
    img = palette[label_raster]

    n_trees = int((np.unique(labels) > 0).sum())
    fig, ax = plt.subplots(figsize=(12, 12), dpi=100)
    ax.imshow(img, origin='lower', extent=[x_min, x_max, y_min, y_max],
              interpolation='nearest')
    ax.set_title(f"{tile_name}  —  {n_trees} trees", fontsize=14)
    ax.set_xlabel("X (m, local)")
    ax.set_ylabel("Y (m, local)")
    plt.tight_layout()
    out = viz_dir / f"{tile_name}_segmentation.png"
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"    seg image → {out}")


# ---------------------------------------------------------------------------
# Mesh helpers
# ---------------------------------------------------------------------------

def segment_to_marching_cubes(pts, voxel_size=0.75, sigma=1.5, iso_level=0.15):
    """Watertight crown mesh via Gaussian-smoothed voxel grid + marching cubes."""
    if len(pts) < 10:
        return np.zeros((0, 3)), np.zeros((0, 3), dtype=int)

    padding = 3.0 * sigma * voxel_size
    min_b = pts.min(axis=0) - padding
    max_b = pts.max(axis=0) + padding
    dims  = np.ceil((max_b - min_b) / voxel_size).astype(int)

    idx = np.clip(np.floor((pts - min_b) / voxel_size).astype(int), 0, dims - 1)
    grid = np.zeros(dims, dtype=np.float32)
    np.add.at(grid, tuple(idx.T), 1.0)

    smoothed = gaussian_filter(grid, sigma=sigma)
    max_d = smoothed.max()
    if max_d == 0:
        return np.zeros((0, 3)), np.zeros((0, 3), dtype=int)
    smoothed /= max_d

    try:
        verts, faces, _, _ = marching_cubes(smoothed, level=min(iso_level, 0.99))
    except ValueError:
        return np.zeros((0, 3)), np.zeros((0, 3), dtype=int)

    verts = verts * voxel_size + min_b
    return verts, faces


def write_obj(path, verts, faces):
    with open(path, "w") as f:
        for v in verts:
            f.write(f"v {v[0]:.6f} {v[1]:.6f} {v[2]:.6f}\n")
        for tri in faces:
            f.write(f"f {tri[0]+1} {tri[1]+1} {tri[2]+1}\n")

In [5]:
def write_colored_ply(path, verts, faces, colors):
    """
    Write a binary PLY file with per-vertex RGB colors.

    verts:  (N, 3) float — vertex positions
    faces:  (F, 3) int   — triangle indices
    colors: (N, 3) uint8 — per-vertex RGB

    Readable by MeshLab, Paraview, Blender, CloudCompare, Rhino.
    """
    n_verts = len(verts)
    n_faces = len(faces)

    header = (
        "ply\n"
        "format binary_little_endian 1.0\n"
        f"element vertex {n_verts}\n"
        "property float x\n"
        "property float y\n"
        "property float z\n"
        "property uchar red\n"
        "property uchar green\n"
        "property uchar blue\n"
        f"element face {n_faces}\n"
        "property list uchar int vertex_indices\n"
        "end_header\n"
    )

    # Pack vertices as structured numpy array (fast, no Python loop)
    v_buf = np.empty(n_verts, dtype=[
        ('x', '<f4'), ('y', '<f4'), ('z', '<f4'),
        ('r',  'u1'), ('g',  'u1'), ('b',  'u1'),
    ])
    v_buf['x'] = verts[:, 0].astype(np.float32)
    v_buf['y'] = verts[:, 1].astype(np.float32)
    v_buf['z'] = verts[:, 2].astype(np.float32)
    v_buf['r'] = colors[:, 0]
    v_buf['g'] = colors[:, 1]
    v_buf['b'] = colors[:, 2]

    # Pack faces as [count=3, v0, v1, v2]
    f_buf = np.empty(n_faces, dtype=[
        ('n', 'u1'), ('v0', '<i4'), ('v1', '<i4'), ('v2', '<i4'),
    ])
    f_buf['n']  = 3
    f_buf['v0'] = faces[:, 0]
    f_buf['v1'] = faces[:, 1]
    f_buf['v2'] = faces[:, 2]

    with open(path, 'wb') as f:
        f.write(header.encode('ascii'))
        f.write(v_buf.tobytes())
        f.write(f_buf.tobytes())

---
## Step 3 — Per-tile: filter → segment → wrap each tree → save combined .obj

In [6]:
tiles = sorted(TILES_DIR.glob("*.laz")) + sorted(TILES_DIR.glob("*.las"))
print(f"Processing {len(tiles)} tiles → {OUTPUT_DIR}/")
print(f"Segmentation images → {VIZ_DIR}/\n")

translation = np.array(TRANSLATION)

with open(OUTPUT_DIR / "offset.txt", "w") as f:
    f.write(f"x_offset = {TRANSLATION[0]}\n")
    f.write(f"y_offset = {TRANSLATION[1]}\n")
    f.write(f"z_offset = {TRANSLATION[2]}\n")
    f.write("# Subtract this from .obj coordinates to recover original UTM\n")

for i, tile in enumerate(tiles, 1):
    m = re.search(r'(\d{6})', tile.stem)
    tile_name = m.group(1) if m else tile.stem
    out_path  = OUTPUT_DIR / f"{tile_name}_chm.obj"
    viz_path  = VIZ_DIR    / f"{tile_name}_segmentation.png"

    if out_path.exists() and viz_path.exists():
        print(f"[{i}/{len(tiles)}] {tile.name}  →  already done, skipping")
        continue

    print(f"[{i}/{len(tiles)}] {tile.name}")

    # 1. Load tile once, extract both vegetation and ground
    with laspy.open(tile) as f:
        las = f.read()
    pts_veg    = filter_tile(las)
    pts_ground = filter_tile_ground(las)
    del las
    print(f"    vegetation (class 5):     {len(pts_veg):>10,} pts")
    print(f"    ground     (class 2):     {len(pts_ground):>10,} pts")

    if len(pts_ground) < 100:
        print("    Too few ground points to build DTM, skipping.\n")
        continue

    # 2. Clean vegetation
    if SOR_NEIGHBORS > 0:
        pts_veg = remove_outliers_sor(pts_veg)
        print(f"    after SOR:                {len(pts_veg):>10,} pts")
    if VOXEL_SIZE > 0:
        pts_veg = voxel_downsample(pts_veg, VOXEL_SIZE)
        print(f"    after voxel downsample:   {len(pts_veg):>10,} pts")

    if len(pts_veg) < 100:
        print("    Too few vegetation points, skipping.\n")
        continue

    pts_veg    -= translation
    pts_ground -= translation

    t0 = time.perf_counter()

    # 3. CHM segmentation — also returns DTM for trunk ground-level lookup
    labels, dtm, (dtm_xmin, dtm_ymin, dtm_cs) = segment_trees_chm(
        pts_veg, pts_ground,
        cell_size       = CHM_CELL_SIZE,
        smooth_sigma    = CHM_SMOOTH_SIGMA,
        min_height      = CHM_MIN_HEIGHT,
        peak_min_dist_m = CHM_PEAK_MIN_DIST,
        min_tree_points = CHM_MIN_TREE_PTS,
    )
    t_seg = time.perf_counter() - t0
    print(f"    segmentation: {t_seg:.1f} s")
    del pts_ground

    # 4. Save segmentation image
    save_segmentation_image(pts_veg, labels, tile_name, VIZ_DIR, cell_size=1.0)

    if out_path.exists():
        print(f"    mesh already exists, skipping meshing.\n")
        del pts_veg, dtm
        continue

    unique_lbls = np.unique(labels[labels > 0])

    # 5. Crown mesh + trunk cone per tree
    all_verts, all_faces = [], []
    vert_offset = 0
    for lbl in unique_lbls:
        tree_pts = pts_veg[labels == lbl]

        # Crown
        v_crown, f_crown = segment_to_marching_cubes(
            tree_pts,
            voxel_size = MC_VOXEL_SIZE,
            sigma      = MC_SIGMA,
            iso_level  = MC_ISO_LEVEL,
        )
        if len(v_crown) == 0:
            continue
        all_verts.append(v_crown)
        all_faces.append(f_crown + vert_offset)
        vert_offset += len(v_crown)

        # Trunk
        tp = estimate_trunk_params(
            tree_pts, dtm, dtm_xmin, dtm_ymin, dtm_cs,
            q_base      = TRUNK_Q_BASE,
            q_cb        = TRUNK_Q_CROWN,
            allom_a     = TRUNK_ALLOM_A,
            allom_b     = TRUNK_ALLOM_B,
            taper_ratio = TRUNK_TAPER,
        )
        if tp is None:
            continue
        v_trunk, f_trunk = make_trunk_mesh(
            tp["x"], tp["y"],
            tp["z_base"], tp["z_crown_base"],
            tp["r_base"], tp["r_top"],
            n_sides = TRUNK_N_SIDES,
        )
        all_verts.append(v_trunk)
        all_faces.append(f_trunk + vert_offset)
        vert_offset += len(v_trunk)

    del pts_veg, dtm

    if not all_verts:
        print("    No meshes produced, skipping.\n")
        continue

    verts   = np.vstack(all_verts)
    faces   = np.vstack(all_faces)
    elapsed = time.perf_counter() - t0

    print(f"    {len(unique_lbls)} trees → {len(verts):,} vertices  "
          f"{len(faces):,} faces  ({elapsed:.1f} s total)")

    write_obj(out_path, verts, faces)
    print(f"    → {out_path}\n")

Processing 25 tiles → bcn_canopy_objs_Li/
Segmentation images → bcn_canopy_objs_Li/segmentation_viz/

[1/25] lidar-territorial-v3r1-full1km428580-2021-2023.laz  →  already done, skipping
[2/25] lidar-territorial-v3r1-full1km428581-2021-2023.laz  →  already done, skipping
[3/25] lidar-territorial-v3r1-full1km428582-2021-2023.laz  →  already done, skipping
[4/25] lidar-territorial-v3r1-full1km428583-2021-2023.laz  →  already done, skipping
[5/25] lidar-territorial-v3r1-full1km428584-2021-2023.laz  →  already done, skipping
[6/25] lidar-territorial-v3r1-full1km429580-2021-2023.laz  →  already done, skipping
[7/25] lidar-territorial-v3r1-full1km429581-2021-2023.laz  →  already done, skipping
[8/25] lidar-territorial-v3r1-full1km429582-2021-2023.laz  →  already done, skipping
[9/25] lidar-territorial-v3r1-full1km429583-2021-2023.laz  →  already done, skipping
[10/25] lidar-territorial-v3r1-full1km429584-2021-2023.laz  →  already done, skipping
[11/25] lidar-territorial-v3r1-full1km430580-20

In [7]:
# ---------------------------------------------------------------------------
# 3-D segmentation visualisation — colored PLY with trunks, one file per tile
# Each tree (crown + trunk) gets a distinct random colour.
# Open in MeshLab / Paraview / Blender / CloudCompare.
#
# Set N_VIZ_TILES to control how many tiles to process.
# Output: bcn_canopy_objs_Li/segmentation_ply/<tile>_segmentation.ply
# ---------------------------------------------------------------------------

N_VIZ_TILES = 6   # change to len(tiles) to process all

ply_dir = OUTPUT_DIR / "segmentation_ply"
ply_dir.mkdir(exist_ok=True)

for i, tile in enumerate(tiles[:N_VIZ_TILES], 1):
    m = re.search(r'(\d{6})', tile.stem)
    tile_name = m.group(1) if m else tile.stem
    ply_path  = ply_dir / f"{tile_name}_segmentation.ply"

    if ply_path.exists():
        print(f"[{i}/{N_VIZ_TILES}] {tile.name}  →  already done, skipping")
        continue

    print(f"[{i}/{N_VIZ_TILES}] {tile.name}")
    t0 = time.perf_counter()

    # Load
    with laspy.open(tile) as f:
        las = f.read()
    pts_veg    = filter_tile(las)
    pts_ground = filter_tile_ground(las)
    del las

    if SOR_NEIGHBORS > 0:
        pts_veg = remove_outliers_sor(pts_veg)
    if VOXEL_SIZE > 0:
        pts_veg = voxel_downsample(pts_veg, VOXEL_SIZE)

    pts_veg    -= translation
    pts_ground -= translation

    # Segment — unpack DTM for trunk estimation
    labels, dtm, (dtm_xmin, dtm_ymin, dtm_cs) = segment_trees_chm(
        pts_veg, pts_ground,
        cell_size       = CHM_CELL_SIZE,
        smooth_sigma    = CHM_SMOOTH_SIGMA,
        min_height      = CHM_MIN_HEIGHT,
        peak_min_dist_m = CHM_PEAK_MIN_DIST,
        min_tree_points = CHM_MIN_TREE_PTS,
    )
    del pts_ground

    unique_lbls = np.unique(labels[labels > 0])

    # Random colour palette — one colour per label
    rng     = np.random.default_rng(42)
    n_lbls  = int(labels.max()) + 1
    palette = rng.integers(60, 230, size=(n_lbls, 3), dtype=np.uint8)

    # Crown + trunk per tree; trunk gets the same colour as its crown
    all_verts, all_faces, all_colors = [], [], []
    vert_offset = 0
    for lbl in unique_lbls:
        tree_pts = pts_veg[labels == lbl]
        color    = palette[lbl]

        # Crown
        v_crown, f_crown = segment_to_marching_cubes(
            tree_pts,
            voxel_size = MC_VOXEL_SIZE,
            sigma      = MC_SIGMA,
            iso_level  = MC_ISO_LEVEL,
        )
        if len(v_crown) == 0:
            continue
        all_verts.append(v_crown)
        all_faces.append(f_crown + vert_offset)
        all_colors.append(np.tile(color, (len(v_crown), 1)))
        vert_offset += len(v_crown)

        # Trunk (same colour)
        tp = estimate_trunk_params(
            tree_pts, dtm, dtm_xmin, dtm_ymin, dtm_cs,
            q_base      = TRUNK_Q_BASE,
            q_cb        = TRUNK_Q_CROWN,
            allom_a     = TRUNK_ALLOM_A,
            allom_b     = TRUNK_ALLOM_B,
            taper_ratio = TRUNK_TAPER,
        )
        if tp is None:
            continue
        v_trunk, f_trunk = make_trunk_mesh(
            tp["x"], tp["y"],
            tp["z_base"], tp["z_crown_base"],
            tp["r_base"], tp["r_top"],
            n_sides = TRUNK_N_SIDES,
        )
        all_verts.append(v_trunk)
        all_faces.append(f_trunk + vert_offset)
        all_colors.append(np.tile(color, (len(v_trunk), 1)))
        vert_offset += len(v_trunk)

    del pts_veg, dtm

    if not all_verts:
        print("    No meshes produced, skipping.")
        continue

    verts  = np.vstack(all_verts)
    faces  = np.vstack(all_faces)
    colors = np.vstack(all_colors).astype(np.uint8)

    write_colored_ply(ply_path, verts, faces, colors)
    elapsed = time.perf_counter() - t0
    print(f"    → {ply_path}")
    print(f"       {len(unique_lbls)} trees  {len(verts):,} verts  "
          f"{len(faces):,} faces  ({elapsed:.1f} s)\n")

[1/6] lidar-territorial-v3r1-full1km428580-2021-2023.laz  →  already done, skipping
[2/6] lidar-territorial-v3r1-full1km428581-2021-2023.laz  →  already done, skipping
[3/6] lidar-territorial-v3r1-full1km428582-2021-2023.laz  →  already done, skipping
[4/6] lidar-territorial-v3r1-full1km428583-2021-2023.laz  →  already done, skipping
[5/6] lidar-territorial-v3r1-full1km428584-2021-2023.laz  →  already done, skipping
[6/6] lidar-territorial-v3r1-full1km429580-2021-2023.laz  →  already done, skipping


In [ ]:
# ============================================================
# City + vegetation flyover video
#
# Requires PyVista for 3-D rendering and video export.
# Install once (if not already):
#   conda install -c conda-forge pyvista
#   conda install -c conda-forge imageio imageio-ffmpeg
# ============================================================

import pyvista as pv

# ---- Colours ----
CROWN_RGB = np.array([ 60, 140,  30], dtype=np.uint8)   # forest green
TRUNK_RGB = np.array([100,  65,  30], dtype=np.uint8)   # bark brown
BLDG_RGB  = (210, 200, 185)                              # warm stone

# ---- Video settings ----
VIDEO_PATH  = OUTPUT_DIR / "barcelona_flyover.mp4"
N_FRAMES    = 360    # 15 s at 24 fps — increase for a slower orbit
FRAMERATE   = 24
WINDOW_W    = 1920
WINDOW_H    = 1080

# ---- City centre (LOCAL coordinates — same as in the .obj/.ply files) ----
# These are coordinates AFTER subtracting TRANSLATION from UTM.
# To find the right value: open any *_chm.obj in MeshLab, pick a point
# at the area you want to orbit around, read its X/Y from the bottom bar.
#
# Quick reference:
#   Geometric centre of the 25-tile grid : (808, -390)
#   To convert from UTM: x_local = utm_easting  - 429692
#                        y_local = utm_northing - 4582890
CITY_CENTER_LOCAL = (808.0, -390.0)   # ← change these two numbers

# ---- Paths ----
BUILDINGS_OBJ = Path("Barcelona/city4cfd/lod2p2_complexity_0p7_Barcelona/Mesh_Buildings.obj")
VEG_PLY_DIR   = OUTPUT_DIR / "vegetation_viz_ply"
VEG_PLY_DIR.mkdir(exist_ok=True)

# ==============================================================
# Step 1 — Build per-tile green-crown / brown-trunk PLY files
# (skipped automatically if the file already exists)
# ==============================================================

for tile_i, tile in enumerate(tiles, 1):
    m = re.search(r'(\d{6})', tile.stem)
    tile_name = m.group(1) if m else tile.stem
    ply_path  = VEG_PLY_DIR / f"{tile_name}_vegetation.ply"

    if ply_path.exists():
        print(f"[{tile_i}/{len(tiles)}] {tile_name}  →  already exists, skipping")
        continue

    print(f"[{tile_i}/{len(tiles)}] {tile_name}  building green/brown PLY...")
    t0 = time.perf_counter()

    with laspy.open(tile) as f:
        las = f.read()
    pts_veg    = filter_tile(las)
    pts_ground = filter_tile_ground(las)
    del las

    if SOR_NEIGHBORS > 0:
        pts_veg = remove_outliers_sor(pts_veg)
    if VOXEL_SIZE > 0:
        pts_veg = voxel_downsample(pts_veg, VOXEL_SIZE)

    pts_veg    -= translation
    pts_ground -= translation

    labels, dtm, (dtm_xmin, dtm_ymin, dtm_cs) = segment_trees_chm(
        pts_veg, pts_ground,
        cell_size       = CHM_CELL_SIZE,
        smooth_sigma    = CHM_SMOOTH_SIGMA,
        min_height      = CHM_MIN_HEIGHT,
        peak_min_dist_m = CHM_PEAK_MIN_DIST,
        min_tree_points = CHM_MIN_TREE_PTS,
    )
    del pts_ground

    unique_lbls = np.unique(labels[labels > 0])
    all_verts, all_faces, all_colors = [], [], []
    vert_offset = 0

    for lbl in unique_lbls:
        tree_pts = pts_veg[labels == lbl]

        v_crown, f_crown = segment_to_marching_cubes(
            tree_pts, voxel_size=MC_VOXEL_SIZE,
            sigma=MC_SIGMA, iso_level=MC_ISO_LEVEL)
        if len(v_crown) == 0:
            continue
        all_verts.append(v_crown)
        all_faces.append(f_crown + vert_offset)
        all_colors.append(np.tile(CROWN_RGB, (len(v_crown), 1)))
        vert_offset += len(v_crown)

        tp = estimate_trunk_params(
            tree_pts, dtm, dtm_xmin, dtm_ymin, dtm_cs,
            q_base=TRUNK_Q_BASE, q_cb=TRUNK_Q_CROWN,
            allom_a=TRUNK_ALLOM_A, allom_b=TRUNK_ALLOM_B,
            taper_ratio=TRUNK_TAPER)
        if tp is None:
            continue
        v_trunk, f_trunk = make_trunk_mesh(
            tp["x"], tp["y"], tp["z_base"], tp["z_crown_base"],
            tp["r_base"], tp["r_top"], n_sides=TRUNK_N_SIDES)
        all_verts.append(v_trunk)
        all_faces.append(f_trunk + vert_offset)
        all_colors.append(np.tile(TRUNK_RGB, (len(v_trunk), 1)))
        vert_offset += len(v_trunk)

    del pts_veg, dtm

    if all_verts:
        write_colored_ply(
            ply_path,
            np.vstack(all_verts),
            np.vstack(all_faces),
            np.vstack(all_colors).astype(np.uint8),
        )
        print(f"    → {ply_path}  ({time.perf_counter() - t0:.1f} s)")
    else:
        print("    No meshes, skipping.")

# ==============================================================
# Step 2 — Assemble scene and render rotating video
# ==============================================================

print("\nAssembling 3-D scene...")
plotter = pv.Plotter(off_screen=True, window_size=(WINDOW_W, WINDOW_H))
plotter.set_background([10, 10, 20])
plotter.enable_eye_dome_lighting()

def _load_colored_ply(path):
    mesh = pv.read(str(path))
    pd   = mesh.point_data
    if "RGB" not in pd and "red" in pd:
        mesh.point_data["RGB"] = np.column_stack(
            [pd["red"], pd["green"], pd["blue"]]).astype(np.uint8)
    return mesh

# --- Vegetation ---
ply_files = sorted(VEG_PLY_DIR.glob("*_vegetation.ply"))
print(f"Loading {len(ply_files)} vegetation tiles...")
for ply_f in ply_files:
    plotter.add_mesh(_load_colored_ply(ply_f), scalars="RGB", rgb=True,
                     smooth_shading=False)

# --- Buildings ---
if BUILDINGS_OBJ.exists():
    print(f"Loading buildings: {BUILDINGS_OBJ}")
    bldg = pv.read(str(BUILDINGS_OBJ))
    bc = np.array(bldg.center)
    if abs(bc[0]) > 10_000 or abs(bc[1]) > 10_000:
        print(f"  UTM → local shift ({bc[0]:.0f}, {bc[1]:.0f})")
        bldg.translate(
            [-float(translation[0]), -float(translation[1]), -float(translation[2])],
            inplace=True)
    plotter.add_mesh(bldg, color=BLDG_RGB, smooth_shading=True)
else:
    print(f"WARNING: buildings file not found:\n  {BUILDINGS_OBJ}")

# --- Camera geometry ---
bounds  = plotter.bounds

# City centre directly in local coordinates (no translation needed)
cx, cy  = CITY_CENTER_LOCAL

scene_r = max(bounds[1] - bounds[0], bounds[3] - bounds[2]) / 2
orbit_r = scene_r * 0.65
cam_z   = bounds[5] + scene_r * 0.35
focus_z = (bounds[4] + bounds[5]) * 0.2

print(f"\nCity centre (local): ({cx:.0f}, {cy:.0f})")
print(f"Orbit radius : {orbit_r:.0f} m  (scene radius {scene_r:.0f} m)")
print(f"Camera height: {cam_z:.0f} m")
print(f"Rendering    : {N_FRAMES} frames × {FRAMERATE} fps = {N_FRAMES/FRAMERATE:.0f} s")

plotter.add_light(pv.Light(
    position=(cx, cy, cam_z * 1.5),
    focal_point=(cx, cy, 0),
    intensity=1.0,
    light_type="scene light",
))

plotter.open_movie(str(VIDEO_PATH), framerate=FRAMERATE, quality=8)

for frame in range(N_FRAMES):
    angle = 2 * np.pi * frame / N_FRAMES
    plotter.camera.position    = (cx + orbit_r * np.cos(angle),
                                   cy + orbit_r * np.sin(angle),
                                   cam_z)
    plotter.camera.focal_point = (cx, cy, focus_z)
    plotter.camera.up          = (0, 0, 1)
    plotter.render()
    plotter.write_frame()
    if frame % 60 == 0:
        print(f"  frame {frame:>4} / {N_FRAMES}")

plotter.close()
print(f"\nVideo saved → {VIDEO_PATH}")